# Email Spam Detection — EDA
Explore and understand your dataset before training.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src importable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data_loader import load_dataset
from src.preprocessor import clean_texts

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load Dataset

In [ ]:
df = load_dataset()
print(df.head())
print(f"\nShape: {df.shape}")

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['label'].value_counts()
axes[0].bar(['Ham (0)', 'Spam (1)'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Ham', 'Spam'], autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.show()

## 3. Text Length Analysis

In [ ]:
df['text_len'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, color, name in [(0, 'steelblue', 'Ham'), (1, 'tomato', 'Spam')]:
    subset = df[df['label'] == label]['text_len']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=name)

axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Word Count')
axes[0].legend()

df.boxplot(column='text_len', by='label', ax=axes[1], 
           boxprops=dict(color='steelblue'))
axes[1].set_title('Word Count by Class')
axes[1].set_xlabel('Label (0=Ham, 1=Spam)')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

print(df.groupby('label')['text_len'].describe())

## 4. Most Common Words

In [ ]:
from collections import Counter

def top_words(df_subset, n=20):
    cleaned = clean_texts(df_subset['text'].tolist(), remove_stopwords=True)
    all_words = ' '.join(cleaned).split()
    return Counter(all_words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, label, color, name in [
    (axes[0], 0, 'steelblue', 'Ham'),
    (axes[1], 1, 'tomato', 'Spam')
]:
    words, counts = zip(*top_words(df[df['label'] == label]))
    ax.barh(words[::-1], counts[::-1], color=color)
    ax.set_title(f'Top Words — {name}')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.show()

## 5. Sample Emails

In [ ]:
print('─'*60)
print('SPAM examples:')
print('─'*60)
for t in df[df['label']==1]['text'].head(3).values:
    print(f'→ {t[:200]}')
    print()

print('─'*60)
print('HAM examples:')
print('─'*60)
for t in df[df['label']==0]['text'].head(3).values:
    print(f'→ {t[:200]}')
    print()